# Astropy：単位・時刻・座標を保つ

対応する本文は [astropy.tex](../chapters/astropy.tex) である。コードセルを上から順にShift+Enterで実行する。各問の説明にある値や図と、セルの下に表示される結果を比べる。ファイルを作る問では、その直前に保存先を用意する。

In [1]:
import astropy.units as u
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import QTable
from astropy.time import Time
from astropy.utils import iers

iers.conf.auto_download = False

## 問1：単位と境界値を含めて判定する

「以内」は境界を含むので `<=` を使う。単位を付けた量と裸の数値を比較すると、非ゼロの数値には単位がないためエラーになる。単位を失った値へ勝手に ns を対応させない。

この後のセルにある `assert` は、結果が指定した条件を満たすかを確かめる命令である。違っていればエラーで止まり、条件を満たせば次へ進む。`np.testing` や `pd.testing` の確認も、配列や表の値が一致しない場合にエラーを出す。画面の数値や図と、自動の確認の両方を見る。

In [2]:
window = 250 * u.ns
dt = np.array([120, 220, 250, 251, 800]) * u.ns
inside = dt <= window
np.testing.assert_array_equal(inside, [True, True, True, False, False])
assert inside.sum() == 3
try:
    _ = dt <= 250
except u.UnitConversionError as exc:
    print(type(exc).__name__)
else:
    raise AssertionError("単位なしの比較を検出できませんでした。")
print(dt[inside])

UnitConversionError
[120. 220. 250.] ns


## 問2：秒境界をまたぐ差を取る

`isot` は T 区切りの日時、`utc` は時刻系を指定する。Time は時刻を2つの数で保持するが、計算誤差は残る。220 ns への照合には0.02 ns の許容差を設定する。単一の float の Unix 秒へ変換してから差を取ると、小さな差を失い得る。

In [3]:
t1 = Time("2026-04-12T11:59:59.999999900", format="isot", scale="utc", precision=9)
t2 = Time("2026-04-12T12:00:00.000000120", format="isot", scale="utc", precision=9)
difference_ns = (t2 - t1).to_value("ns")
assert np.isclose(difference_ns, 220.0, rtol=0, atol=0.02)
assert difference_ns <= window.to_value(u.ns)
print("Time の差 [ns]:", difference_ns)
print("単一 float の Unix 秒の差 [ns]:", (t2.unix - t1.unix) * 1e9)

Time の差 [ns]: 220.00001820288162
単一 float の Unix 秒の差 [ns]: 238.4185791015625


## 問3：UTC と TAI の同じ瞬間を確認する

UTC はうるう秒を含む時刻系で、TAI は原子時である。文字列のラベルが異なっても同じ瞬間を表せる。時刻系を付けたまま減算すると、その同一性を確かめられる。

In [4]:
print("UTC:", t2.utc.iso)
print("TAI:", t2.tai.iso)
assert np.isclose((t2.utc - t2.tai).to_value("ns"), 0, rtol=0, atol=0.02)

UTC: 2026-04-12 12:00:00.000000120
TAI: 2026-04-12 12:00:37.000000120


## 問4：ECSV と FITS を読み戻す

表の time は絶対日時ではなく、ここでは基準からの経過時間 [ns] を表す。ECSV は列単位を保持し、FITS の画像では `BUNIT` に信号の単位を記録する。`overwrite=True` はこの実習の出力先にある同名ファイルを置換する。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/10_astropy/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [5]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "10_astropy"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/10_astropy


In [6]:
table = QTable({"time": [0, 120, 250] * u.ns,
                "signal": [12.4, 9.8, 15.1] * u.mV,
                "detector": ["A", "B", "A"]})
ecsv = WORK / "events.ecsv"
table.write(ecsv, format="ascii.ecsv", overwrite=True)
restored = QTable.read(ecsv, format="ascii.ecsv")
assert len(restored) == 3 and restored["time"].unit == u.ns and restored["signal"].unit == u.mV
np.testing.assert_array_equal(restored["signal"].to_value(u.mV), table["signal"].to_value(u.mV))
hdu = fits.PrimaryHDU(table["signal"].to_value(u.mV))
hdu.header["BUNIT"] = "mV"
hdu.writeto(WORK / "signal.fits", overwrite=True)
with fits.open(WORK / "signal.fits") as hdul:
    assert hdul[0].data.shape == (3,)
    assert u.Unit(hdul[0].header["BUNIT"]) == restored["signal"].unit
    np.testing.assert_array_equal(hdul[0].data, restored["signal"].to_value(u.mV))
print(restored)

 time signal detector
  ns    mV           
----- ------ --------
  0.0   12.4        A
120.0    9.8        B
250.0   15.1        A


## 問5：座標系を明示して変換する

`ra` は赤経、`dec` は赤緯、`frame` は入力の座標系である。ICRS から銀河座標への変換には方向を使う。地平座標は観測者から見た方向なので、観測時刻と観測場所も必要となる。

In [7]:
coord = SkyCoord(ra=150*u.deg, dec=20*u.deg, frame="icrs")
galactic = coord.galactic
print("銀河経度 [deg]:", galactic.l.deg, "銀河緯度 [deg]:", galactic.b.deg)
np.testing.assert_allclose([galactic.l.deg, galactic.b.deg], [213.800977, 50.263718], rtol=0, atol=1e-6)

銀河経度 [deg]: 213.80097735289047 銀河緯度 [deg]: 50.26371770865176
